In [1]:
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("laotse/credit-risk-dataset")
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "credit_risk_dataset.csv"))
print(df.shape)
print(df.head())
print(df.isna().sum())
print(df["loan_status"].value_counts(normalize=True))

['credit_risk_dataset.csv']
(32581, 12)
   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                 

In [2]:
print(df["person_age"].describe())
print(df["person_emp_length"].describe())
print((df["person_age"] > 100).sum(), (df["person_emp_length"] > 60).sum())

count    32581.000000
mean        27.734600
std          6.348078
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max        144.000000
Name: person_age, dtype: float64
count    31686.000000
mean         4.789686
std          4.142630
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max        123.000000
Name: person_emp_length, dtype: float64
5 2


In [3]:
# --- Section 1: cleaning ---
# Rule: only remove rows that are impossible or exact duplicates.
# Missing values (emp_length, int_rate) are KEPT as NaN here and imputed later
# inside each CV fold, so no statistics from test data leak into training.
log = {"start": len(df)}

# 1. Exact duplicate rows (same applicant + same loan recorded twice)
df = df.drop_duplicates()
log["after_drop_duplicates"] = len(df)

# 2. Impossible ages (e.g. 144 years old)
df = df[df["person_age"] <= 100]
log["after_age_filter"] = len(df)

# 3. Impossible employment length: > 60 years, or longer than the person could
#    have worked (started before age 14). NaN rows are kept (comparison is False).
bad_emp = (df["person_emp_length"] > 60) | (df["person_emp_length"] > df["person_age"] - 14)
df = df[~bad_emp]
log["after_emp_filter"] = len(df)

df = df.reset_index(drop=True)
for step, n in log.items():
    print(f"{step:<24}{n:>7}")
print("\nRows removed:", log["start"] - len(df))
print("\nMissing values still present (handled later in pipelines):")
print(df.isna().sum()[lambda s: s > 0])
print("\nDefault rate after cleaning:", round(df["loan_status"].mean(), 4))

start                     32581
after_drop_duplicates     32416
after_age_filter          32411
after_emp_filter          32409

Rows removed: 172

Missing values still present (handled later in pipelines):
person_emp_length     887
loan_int_rate        3094
dtype: int64

Default rate after cleaning: 0.2187


In [2]:
# --- Section 2: train/test split (80/20, stratified) ---
# The test set is created ONCE here and saved, so every later notebook
# (classification and regression) is evaluated on exactly the same unseen rows.
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
OUT = Path.cwd().parent / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["loan_status"],   # keep the ~22% default rate in both halves
    random_state=RANDOM_STATE,    # same split every time the notebook runs
)

train_df.to_csv(OUT / "train.csv", index=False)
test_df.to_csv(OUT / "test.csv", index=False)

print(f"train: {train_df.shape}   test: {test_df.shape}")
print(f"default rate  train: {train_df['loan_status'].mean():.4f}   test: {test_df['loan_status'].mean():.4f}")
print("missing int_rate  train:", train_df["loan_int_rate"].isna().sum(), "  test:", test_df["loan_int_rate"].isna().sum())
print("saved to", OUT)

NameError: name 'df' is not defined